# stg_validation

## 0. Setup & Connection

In [14]:
import os
import re
import json
import pandas as pd
import snowflake.connector
from dotenv import load_dotenv
from IPython.display import display

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 100)

load_dotenv()

conn = snowflake.connector.connect(
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    role=os.getenv("SNOWFLAKE_ROLE"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    database=os.getenv("SNOWFLAKE_DATABASE"),
)

def run_query(sql: str) -> pd.DataFrame:
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [desc[0] for desc in cur.description]
    cur.close()
    return pd.DataFrame(rows, columns=cols)

print("Connected!")

Connected!


## 1. JSearch

In [15]:
raw_df = run_query("SELECT SOURCE, RAW_PAYLOAD::STRING AS payload, INGESTED_AT FROM RAW.JSEARCH.SRC_POSTINGS")
print(f"Raw rows: {len(raw_df)}")
print(f"Columns: {raw_df.columns.tolist()}")

parsed = []
for _, row in raw_df.iterrows():
    p = json.loads(row['PAYLOAD'])
    parsed.append({
        # --- identity ---
        'job_id':           p.get('job_id'),
        'source_raw':       row['SOURCE'],
        'ingested_at':      row['INGESTED_AT'],

        # --- core fields ---
        'job_title':        p.get('job_title'),
        'company_name':     p.get('employer_name'),
        'job_url':          p.get('job_apply_link'),
        'date_posted':      p.get('job_posted_at_datetime_utc'),
        'description':      p.get('job_description'),

        # --- location ---
        'city':             p.get('job_city'),
        'state':            p.get('job_state'),
        'country':          p.get('job_country'),
        'latitude':         p.get('job_latitude'),
        'longitude':        p.get('job_longitude'),

        # --- employment ---
        'employment_type_raw': p.get('job_employment_type'),

        # --- salary (raw) ---
        'salary_min_raw':   p.get('job_min_salary'),
        'salary_max_raw':   p.get('job_max_salary'),
        'salary_period':    p.get('job_salary_period'),

        # --- remote signal (broken boolean, title is real signal) ---
        'job_is_remote':    p.get('job_is_remote'),
    })

stg = pd.DataFrame(parsed)
print(f"Parsed rows: {len(stg)}")

Raw rows: 136
Columns: ['SOURCE', 'PAYLOAD', 'INGESTED_AT']
Parsed rows: 136


In [16]:
# ── 3. Normalize source ───────────────────────────────────────────────────────
# Raw SOURCE looks like "jsearch:Data Analyst in New York" — strip to just "jsearch"
stg['source'] = 'jsearch'

# ── 4. Normalize date_posted to date only (drop time component) ───────────────
stg['date_posted'] = pd.to_datetime(stg['date_posted'], utc=True).dt.date

# ── 5. Normalize employment_type ─────────────────────────────────────────────
# JSearch returns "Full-time", "Part-time", "Contractor" etc.
def normalize_employment_type(val) -> str:
    if not val or not isinstance(val, str):
        return None
    v = val.lower().strip()
    if 'full' in v:                          return 'full_time'
    if 'part' in v:                          return 'part_time'
    if 'contract' in v or 'contractor' in v: return 'contract'
    return 'other'

stg['employment_type'] = stg['employment_type_raw'].apply(normalize_employment_type)

print("employment_type value counts:")
print(stg['employment_type'].value_counts(dropna=False))

employment_type value counts:
employment_type
full_time    119
contract      10
other          4
part_time      2
NaN            1
Name: count, dtype: int64


In [17]:
# ── 6. Derive work_model from title text ──────────────────────────────────────
# job_is_remote is broken — only fires TRUE for "Anywhere" listings
# Real signal is in the job title

REMOTE_RE = re.compile(r'\bremote\b',  re.IGNORECASE)
HYBRID_RE  = re.compile(r'\bhybrid\b', re.IGNORECASE)

def derive_work_model(row) -> str:
    title = row['job_title'] or ''
    if REMOTE_RE.search(title):  return 'remote'
    if HYBRID_RE.search(title):  return 'hybrid'
    if row['job_is_remote']:     return 'remote'   # fallback: "Anywhere" listings
    return 'onsite'

stg['work_model'] = stg.apply(derive_work_model, axis=1)

print("\nwork_model value counts:")
print(stg['work_model'].value_counts(dropna=False))


work_model value counts:
work_model
onsite    112
remote     19
hybrid      5
Name: count, dtype: int64


In [18]:
# ── 7. Salary — null out hourly, keep annual only ─────────────────────────────
stg['salary_min'] = stg.apply(
    lambda r: r['salary_min_raw'] if r['salary_period'] == 'YEAR' else None, axis=1
)
stg['salary_max'] = stg.apply(
    lambda r: r['salary_max_raw'] if r['salary_period'] == 'YEAR' else None, axis=1
)

print("Salary coverage after hourly filter:")
print(f"  salary_min populated: {stg['salary_min'].notna().sum()} / {len(stg)}")
print(f"  Hourly rows nulled out: {(stg['salary_period'] == 'HOUR').sum()}")

Salary coverage after hourly filter:
  salary_min populated: 37 / 136
  Hourly rows nulled out: 4


In [19]:
# ── 8. Senior title filter ────────────────────────────────────────────────────
SENIOR_RE = re.compile(
    r'\b(senior|sr\.?|lead|principal|staff|manager|director|vp|vice president|'
    r'avp|head of|architect|chief|svp|evp|gvp|president|officer|executive|leader)\b',
    re.IGNORECASE
)

# Malformed listings — title is just a company name or otherwise nonsensical
MALFORMED_RE = re.compile(
    r'^(orbis|owner$)',
    re.IGNORECASE
)

stg['is_senior'] = stg['job_title'].apply(
    lambda t: bool(SENIOR_RE.search(t)) if t else False
)
stg['is_malformed'] = stg['job_title'].apply(
    lambda t: bool(MALFORMED_RE.search(t)) if t else False
)

print(f"Senior titles flagged: {stg['is_senior'].sum()}")
print(f"Malformed titles flagged: {stg['is_malformed'].sum()}")

Senior titles flagged: 52
Malformed titles flagged: 2


In [20]:
# ── 9. Deduplicate within JSearch — keep most recent ingested_at per job_id ───
before = len(stg)
stg = (
    stg.sort_values('ingested_at', ascending=False)
       .drop_duplicates(subset='job_id', keep='first')
       .reset_index(drop=True)
)
after = len(stg)
print(f"Rows before dedup: {before}")
print(f"Rows after dedup:  {after}")
print(f"Duplicates removed: {before - after}")

Rows before dedup: 136
Rows after dedup:  133
Duplicates removed: 3


In [21]:
# ── 10. Apply all filters and select final columns ────────────────────────────
stg_final = (
    stg[~stg['is_senior'] & ~stg['is_malformed']]
    [[
        'job_id', 'source', 'job_title', 'company_name', 'job_url',
        'date_posted', 'description', 'city', 'state', 'country',
        'latitude', 'longitude', 'work_model', 'employment_type',
        'salary_min', 'salary_max', 'ingested_at',
    ]]
    .reset_index(drop=True)
)

print(f"Final stg_jsearch rows: {len(stg_final)}")
print(f"\nNull rates:")
null_rates = (stg_final.isna().sum() / len(stg_final) * 100).round(1)
print(null_rates.to_string())
print(f"\nSample output:")
display(stg_final.head(10))

Final stg_jsearch rows: 82

Null rates:
job_id              0.0
source              0.0
job_title           0.0
company_name        0.0
job_url             0.0
date_posted         0.0
description         0.0
city                8.5
state               8.5
country             8.5
latitude            8.5
longitude           8.5
work_model          0.0
employment_type     1.2
salary_min         76.8
salary_max         76.8
ingested_at         0.0

Sample output:


,job_id,source,job_title,company_name,job_url,date_posted,description,city,state,country,latitude,longitude,work_model,employment_type,salary_min,salary_max,ingested_at
0,F6PYvHvaccZ-7SSAAAAAAA==,jsearch,Data Scientist Analyst - Secondaries & Primaries,Ardian,https://www.linkedin.com/jobs/view/data-scientist-analyst-secondaries-primar...,2026-06-02,The Role\n\nThis position offers a unique opportunity to immerse yourself in...,New York,New York,US,40.712775,-74.005973,onsite,full_time,100000.0,120000.0,2026-06-04 12:04:35.529025+00:00
1,n3c20NUW3uecFt_gAAAAAA==,jsearch,Payment Integrity Analyst - Data Mining & Savings Reporting,MVP Health Care,https://www.linkedin.com/jobs/view/payment-integrity-analyst-data-mining-sav...,2026-06-03,"Join Us in Shaping the Future of Health Care\n\nAt MVP Health Care, we're on...",NaN,NaN,NaN,NaN,NaN,remote,full_time,NaN,NaN,2026-06-04 12:04:35.529025+00:00
2,sgF_XtGH7nCvIqGiAAAAAA==,jsearch,"Analytics Engineer (TakeUp), Mid-Level (Remote)",jobright.com,https://www.monster.com/job-openings/analytics-engineer-takeup-mid-level-rem...,2026-06-02,"Join to apply for the Analytics Engineer (TakeUp), Mid-Level (Remote) role a...",NaN,NaN,NaN,NaN,NaN,remote,full_time,NaN,NaN,2026-06-04 12:04:35.529025+00:00
3,AiGTSWE65L2dSiWUAAAAAA==,jsearch,"Analytics Engineer, Service Ops Analytics & AI",Capgemini,https://www.linkedin.com/jobs/view/analytics-engineer-service-ops-analytics-...,2026-06-02,The goal of analytics engineering team within the Service Analytics and AI o...,New York,New York,US,40.712775,-74.005973,onsite,full_time,NaN,NaN,2026-06-04 12:04:35.529025+00:00
4,l9ZA99eUKMJgGhKhAAAAAA==,jsearch,Digital Marketing Analyst,Blinds To Go,https://www.linkedin.com/jobs/view/digital-marketing-analyst-at-blinds-to-go...,2026-06-03,Key member of the marketing team responsible for managing the digital market...,Paramus,New Jersey,US,40.948279,-74.067277,onsite,full_time,80000.0,100000.0,2026-06-04 12:04:35.529025+00:00
5,TGoFXZxdzzYut2LoAAAAAA==,jsearch,Toxicology Analytics Intern,Acutis Diagnostics,https://www.linkedin.com/jobs/view/toxicology-analytics-intern-at-acutis-dia...,2026-06-03,Acutis Diagnostics is seeking a highly motivated and analytical intern to su...,Hicksville,New York,US,40.773079,-73.527900,onsite,other,NaN,NaN,2026-06-04 12:04:35.529025+00:00
6,qjY4hQcRsqcjnK0oAAAAAA==,jsearch,Data Scientist - ML and Advanced Analytics,Soni,https://www.linkedin.com/jobs/view/data-scientist-ml-and-advanced-analytics-...,2026-06-03,Our client is seeking a Data Scientist with deep expertise in machine learni...,Hazlet,New Jersey,US,40.421014,-74.185058,onsite,full_time,135000.0,180000.0,2026-06-04 12:04:35.529025+00:00
7,FC3VorrIevRz6m2wAAAAAA==,jsearch,Rebate Analyst,Celltrion USA,https://www.linkedin.com/jobs/view/rebate-analyst-at-celltrion-usa-4424246939,2026-06-03,Celltrion USA is Celltrion’s U.S. subsidiary established in 2018. Headquarte...,Jersey City,New Jersey,US,40.719534,-74.043013,onsite,full_time,NaN,NaN,2026-06-04 12:04:35.529025+00:00
8,ylm4WjZ5VJtXqZDWAAAAAA==,jsearch,MIM - Analyst - Data and Reporting,MetLife,https://www.linkedin.com/jobs/view/mim-analyst-data-and-reporting-at-metlife...,2026-06-01,Requirements\n\nDescription and Requirements\n\nThe Team You Will Join\n\nMe...,Hanover,New Jersey,US,40.829679,-74.433046,onsite,full_time,NaN,NaN,2026-06-04 12:04:35.529025+00:00
9,tm0JdaNzVIpbpKCvAAAAAA==,jsearch,Policy Analyst (Hybrid schedule) (49822),Care For the Homeless,https://www.linkedin.com/jobs/view/policy-analyst-hybrid-schedule-49822-at-c...,2026-06-03,Summary\n\nThe Policy Analyst is responsible for advancing CFH’s legislative...,New York,New York,US,40.712775,-74.005973,hybrid,full_time,NaN,NaN,2026-06-04 12:04:35.529025+00:00


In [22]:
print("All titles in final stg_jsearch output:")
for t in sorted(stg_final['job_title'].tolist()):
    print(t)

All titles in final stg_jsearch output:
Adobe Customer Journey Analytics Engineer - Remote 35
Analyst, Strategy & Operations
Analytics Engineer (TakeUp), Mid-Level (Remote)
Analytics Engineer - Sports Data & Modeling (Hybrid)
Analytics Engineer, Service Ops Analytics & AI
Azure Databricks Platform Engineer
Billing Analyst: Data-Driven Invoicing & Compliance
Business Analyst III AMZ26810.1
Business Intelligence Engineer, Ad Sales Finance Analytics
Business Intelligence Engineer, Rapid & Rural Logistics (R2L) Science & AI
Capital Markets Data Engineer: Real-Time Analytics & Automation
Credit Business Analyst, Growth
Data Analyst
Data Analyst (Entry-Level / Junior)
Data Analyst (New York)
Data Analyst (R, Excel, Stata, Adobe)
Data Analyst ,Purchase, NY; Florham Park, NJ; Remote will be considered- Immediate position
Data Analyst - Economic Insights & Communications
Data Analyst 1 - 32409
Data Analyst I
Data Analyst | Remote
Data Analyst, Operations Planning
Data Analyst, Operations Planni

## 2. TheirStack

In [23]:
# ── stg_theirstack ────────────────────────────────────────────────────────────
raw_df = run_query("SELECT SOURCE, RAW_PAYLOAD::STRING AS payload, INGESTED_AT FROM RAW.THEIRSTACK.SRC_POSTINGS")
print(f"Raw rows: {len(raw_df)}")
print(f"Columns: {raw_df.columns.tolist()}")

parsed = []
for _, row in raw_df.iterrows():
    p = json.loads(row['PAYLOAD'])
    parsed.append({
        # --- identity ---
        'job_id':           str(p.get('id')),
        'source_raw':       row['SOURCE'],
        'ingested_at':      row['INGESTED_AT'],

        # --- core fields ---
        'job_title':        p.get('job_title'),
        'company_name':     p.get('company'),
        'job_url':          p.get('url'),
        'date_posted':      p.get('date_posted'),
        'description':      p.get('description'),

        # --- location ---
        'city':             p.get('short_location', '').split(',')[0].strip() if p.get('short_location') else None,
        'state':            p.get('state_code'),
        'country':          p.get('country_code'),
        'latitude':         p.get('latitude'),
        'longitude':        p.get('longitude'),

        # --- employment ---
        'employment_type_raw': p.get('employment_statuses'),

        # --- work model (reliable booleans) ---
        'remote_raw':       p.get('remote'),
        'hybrid_raw':       p.get('hybrid'),

        # --- salary (already annual USD) ---
        'salary_min_raw':   p.get('min_annual_salary_usd'),
        'salary_max_raw':   p.get('max_annual_salary_usd'),
    })

stg_ts = pd.DataFrame(parsed)
print(f"Parsed rows: {len(stg_ts)}")

Raw rows: 18
Columns: ['SOURCE', 'PAYLOAD', 'INGESTED_AT']
Parsed rows: 18


In [24]:
# Normalize source
stg_ts['source'] = 'theirstack'

# Normalize date
stg_ts['date_posted'] = pd.to_datetime(stg_ts['date_posted'], utc=True).dt.date

# Employment type — TheirStack returns a list e.g. ['full_time']
def normalize_ts_employment(val) -> str:
    if not val or not isinstance(val, list):
        return None
    v = val[0].lower() if val else None
    if not v:
        return None
    if 'full' in v:    return 'full_time'
    if 'part' in v:    return 'part_time'
    if 'contract' in v: return 'contract'
    return 'other'

stg_ts['employment_type'] = stg_ts['employment_type_raw'].apply(normalize_ts_employment)

# Work model — remote and hybrid booleans are reliable on TheirStack
def derive_ts_work_model(row) -> str:
    if row['remote_raw']:  return 'remote'
    if row['hybrid_raw']:  return 'hybrid'
    return 'onsite'

stg_ts['work_model'] = stg_ts.apply(derive_ts_work_model, axis=1)

# Salary — already annual USD, no filtering needed
stg_ts['salary_min'] = stg_ts['salary_min_raw']
stg_ts['salary_max'] = stg_ts['salary_max_raw']

print("employment_type value counts:")
print(stg_ts['employment_type'].value_counts(dropna=False))
print("\nwork_model value counts:")
print(stg_ts['work_model'].value_counts(dropna=False))
print("\nSalary coverage:")
print(f"  salary_min populated: {stg_ts['salary_min'].notna().sum()} / {len(stg_ts)}")

employment_type value counts:
employment_type
full_time    15
contract      2
NaN           1
Name: count, dtype: int64

work_model value counts:
work_model
onsite    9
hybrid    7
remote    2
Name: count, dtype: int64

Salary coverage:
  salary_min populated: 2 / 18


In [25]:
# Senior title filter — same regex as JSearch
SENIOR_RE = re.compile(
    r'\b(senior|sr\.?|lead|principal|staff|manager|director|vp|vice president|'
    r'avp|head of|architect|chief|svp|evp|gvp|president|officer|executive|leader)\b',
    re.IGNORECASE
)

stg_ts['is_senior'] = stg_ts['job_title'].apply(
    lambda t: bool(SENIOR_RE.search(t)) if t else False
)

print(f"Senior titles flagged: {stg_ts['is_senior'].sum()} / {len(stg_ts)}")
print(stg_ts[stg_ts['is_senior']]['job_title'].tolist())

Senior titles flagged: 0 / 18
[]


In [26]:
# Dedup within TheirStack (shouldn't be any but good hygiene)
before = len(stg_ts)
stg_ts = (
    stg_ts.sort_values('ingested_at', ascending=False)
          .drop_duplicates(subset='job_id', keep='first')
          .reset_index(drop=True)
)
print(f"Rows before dedup: {before} | after: {len(stg_ts)} | removed: {before - len(stg_ts)}")

Rows before dedup: 18 | after: 18 | removed: 0


In [27]:
# Apply filters and select final columns
stg_ts_final = (
    stg_ts[~stg_ts['is_senior']]
    [[
        'job_id', 'source', 'job_title', 'company_name', 'job_url',
        'date_posted', 'description', 'city', 'state', 'country',
        'latitude', 'longitude', 'work_model', 'employment_type',
        'salary_min', 'salary_max', 'ingested_at',
    ]]
    .reset_index(drop=True)
)

print(f"Final stg_theirstack rows: {len(stg_ts_final)}")
print(f"\nNull rates:")
print((stg_ts_final.isna().sum() / len(stg_ts_final) * 100).round(1).to_string())
print(f"\nSample output:")
display(stg_ts_final.head(10))

Final stg_theirstack rows: 18

Null rates:
job_id              0.0
source              0.0
job_title           0.0
company_name        0.0
job_url             0.0
date_posted         0.0
description         0.0
city                0.0
state               0.0
country             0.0
latitude            0.0
longitude           0.0
work_model          0.0
employment_type     5.6
salary_min         88.9
salary_max         88.9
ingested_at         0.0

Sample output:


,job_id,source,job_title,company_name,job_url,date_posted,description,city,state,country,latitude,longitude,work_model,employment_type,salary_min,salary_max,ingested_at
0,704913940,theirstack,Data Analyst-Institute for Advanced Medicine-Aids Center Peter Krueger Clini...,Mount Sinai Morningside,https://www.linkedin.com/jobs/view/data-analyst-institute-for-advanced-medic...,2026-06-03,**Description**\nJob Description\n \n \n**Position Title**\n**Data Analyst...,New York,NY,US,40.714270,-74.00597,onsite,full_time,NaN,NaN,2026-06-04 12:04:35.529034+00:00
1,704438351,theirstack,"Data Analyst, Center of Surgical and Transplant Applied Research (CSTAR)",NYU Langone Health,https://www.linkedin.com/jobs/view/data-analyst-center-of-surgical-and-trans...,2026-06-03,1153308\_RR00113253 Job ID: 1153308\_RR00113253\n \n \nNYU Grossman School...,New York,NY,US,40.714270,-74.00597,onsite,full_time,NaN,NaN,2026-06-04 12:04:35.529034+00:00
2,702956448,theirstack,GIS Data Analyst,Trident Consulting,https://www.linkedin.com/jobs/view/gis-data-analyst-at-trident-consulting-44...,2026-06-02,Trident Consulting is seeking a “\n**GIS Data Analyst**\n” for one of our cl...,New York,NY,US,40.714270,-74.00597,onsite,contract,NaN,NaN,2026-06-04 12:04:35.529034+00:00
3,703117190,theirstack,"Analytics Engineer, Service Ops Analytics & AI",Capgemini,https://www.linkedin.com/jobs/view/analytics-engineer-service-ops-analytics-...,2026-06-02,The goal of analytics engineering team within the Service Analytics and AI o...,New York,NY,US,40.714270,-74.00597,onsite,full_time,NaN,NaN,2026-06-04 12:04:35.529034+00:00
4,704952793,theirstack,Data Analyst,ATC,https://www.linkedin.com/jobs/view/data-analyst-at-atc-4424220166,2026-06-03,**About Us:**\n\nAmerican Technology Consulting (ATC) is a service-first tec...,New York,NY,US,40.714270,-74.00597,onsite,full_time,NaN,NaN,2026-06-04 12:04:35.529034+00:00
5,705349326,theirstack,Economic Data Analyst,Toll International LLC,https://www.linkedin.com/jobs/view/economic-data-analyst-at-toll-internation...,2026-06-03,Help shape the economic insights that support some of the most critical tran...,New York,NY,US,40.714270,-74.00597,hybrid,full_time,NaN,NaN,2026-06-04 12:04:35.529034+00:00
6,704607328,theirstack,Data Analyst,Hanvok Group LLC,https://www.linkedin.com/jobs/view/data-analyst-at-hanvok-group-llc-4424091714,2026-06-03,**Company Description**\nHanvok Group LLC helps professionals and businesses...,New York,NY,US,40.714270,-74.00597,remote,full_time,NaN,NaN,2026-06-04 12:04:35.529034+00:00
7,703949131,theirstack,Data Analyst,ThesisGrid,http://www.indeed.com/job/data-analyst-1ace763e7e33acce,2026-06-03,About ThesisGrid Capital\n\nThesisGrid Capital is an AI-native investment re...,New York,NY,US,40.762188,-73.97265,onsite,full_time,70000.0,120000.0,2026-06-04 12:04:35.529034+00:00
8,702699905,theirstack,Data Analyst,edkey,https://www.linkedin.com/jobs/view/data-analyst-at-edkey-4423957473,2026-06-02,**About the Role**\n\nWe are hiring a Data Analyst on the Product Data Scien...,New York,NY,US,40.714270,-74.00597,hybrid,full_time,NaN,NaN,2026-06-04 12:04:35.529034+00:00
9,704833094,theirstack,Data Analyst (Entry-Level / Junior),Africa Explorer,https://www.linkedin.com/jobs/view/data-analyst-entry-level-junior-at-africa...,2026-06-03,**Data Analyst (Entry-Level / Junior)**\n\nRole Description\n\nAnalyze and i...,New York,NY,US,40.714270,-74.00597,hybrid,full_time,NaN,NaN,2026-06-04 12:04:35.529034+00:00


In [28]:
print(f"\nAll titles:")
for t in sorted(stg_ts_final['job_title'].tolist()):
    print(t)


All titles:
Advertising Data Analyst
Analytics Engineer, Service Ops Analytics & AI
Data Analyst
Data Analyst
Data Analyst
Data Analyst
Data Analyst
Data Analyst (Entry-Level / Junior)
Data Analyst (Entry-Level / Junior)
Data Analyst, Center of Surgical and Transplant Applied Research (CSTAR)
Data Analyst-Institute for Advanced Medicine-Aids Center Peter Krueger Clinic-Mount Sinai Health System-Full Time-Days
Economic Data Analyst
Economic Data Analyst (Transportation & Infrastructure)
GIS Data Analyst
Junior Data Analyst w/ Workday Exp....Rate-$30/hronC2C....Locals Only
Market Data Analyst
Product Data Analyst
SQL Data Analyst


## 3. Built In NYC

In [29]:
# ── stg_builtin ───────────────────────────────────────────────────────────────
raw_df = run_query("SELECT SOURCE, RAW_PAYLOAD::STRING AS payload, INGESTED_AT FROM RAW.BUILTIN.SRC_POSTINGS")
print(f"Raw rows: {len(raw_df)}")
print(f"Columns: {raw_df.columns.tolist()}")

parsed = []
for _, row in raw_df.iterrows():
    p = json.loads(row['PAYLOAD'])

    # --- nested object extractions ---
    identifier       = p.get('identifier') or {}
    hiring_org       = p.get('hiringOrganization') or {}
    job_location     = p.get('jobLocation') or {}
    address          = job_location.get('address') or {}
    geo              = job_location.get('geo') or {}
    base_salary      = p.get('baseSalary') or {}
    salary_value     = base_salary.get('value') or {}

    parsed.append({
        # --- identity ---
        'job_id':           str(identifier.get('value')),
        'source_raw':       row['SOURCE'],
        'ingested_at':      row['INGESTED_AT'],

        # --- core fields ---
        'job_title':        p.get('title'),
        'company_name':     hiring_org.get('name'),
        'job_url':          p.get('source_url'),
        'date_posted':      p.get('datePosted'),
        'description_raw':  p.get('description'),   # HTML — will strip below

        # --- location ---
        'city':             address.get('addressLocality'),
        'state':            address.get('addressRegion'),
        'country':          address.get('addressCountry'),
        'latitude':         geo.get('latitude'),
        'longitude':        geo.get('longitude'),

        # --- employment ---
        'employment_type_raw': p.get('employmentType'),

        # --- work model ---
        'job_location_type': p.get('jobLocationType'),   # 'TELECOMMUTE' = remote

        # --- salary ---
        'salary_min_raw':   salary_value.get('minValue'),
        'salary_max_raw':   salary_value.get('maxValue'),
        'salary_unit':      salary_value.get('unitText'),  # filter to YEAR only
    })

stg_bi = pd.DataFrame(parsed)
print(f"Parsed rows: {len(stg_bi)}")

Raw rows: 18
Columns: ['SOURCE', 'PAYLOAD', 'INGESTED_AT']
Parsed rows: 18


In [37]:
import re
import html

# ── HTML stripping ────────────────────────────────────────────────────────────
# Built In descriptions are raw HTML — strip all tags and normalize whitespace
HTML_TAG_RE    = re.compile(r'<[^>]+>')
WHITESPACE_RE  = re.compile(r'\s+')

def strip_html(val) -> str:
    if not val or not isinstance(val, str):
        return None
    text = HTML_TAG_RE.sub(' ', val)
    text = html.unescape(text)          # converts &nbsp; &amp; &gt; etc.
    text = WHITESPACE_RE.sub(' ', text).strip()
    return text

stg_bi['description'] = stg_bi['description_raw'].apply(strip_html)

# Sanity check — show raw vs cleaned for one row
print("RAW:")
print(stg_bi['description_raw'].iloc[0][:300])
print("\nCLEANED:")
print(stg_bi['description'].iloc[0][:300])

RAW:
<p><strong>Job Title:</strong> Data Analyst</p><p><strong>Location:&nbsp;</strong>New York, NY</p><p><strong>Start Date</strong>: ASAP</p><b>&nbsp;</b><p>Whalar is the leading, most awarded, independent Creator and Social agency.&nbsp; We transform brands into cultural drivers by unlocking the full 

CLEANED:
Job Title: Data Analyst Location: New York, NY Start Date : ASAP Whalar is the leading, most awarded, independent Creator and Social agency. We transform brands into cultural drivers by unlocking the full creative power of Creators. We go beyond the conventional social and influencer strategy. We ha


In [43]:
# ── Normalize source ──────────────────────────────────────────────────────────
stg_bi['source'] = 'builtin'

# ── Normalize date ────────────────────────────────────────────────────────────
stg_bi['date_posted'] = pd.to_datetime(stg_bi['date_posted'], utc=True).dt.date

# ── Employment type ───────────────────────────────────────────────────────────
# Built In returns "FULL_TIME", "PART_TIME", "CONTRACTOR" etc.
def normalize_bi_employment(val) -> str:
    if not val or not isinstance(val, str):
        return None
    v = val.lower().strip()
    if 'full' in v:     return 'full_time'
    if 'part' in v:     return 'part_time'
    if 'contract' in v: return 'contract'
    return 'other'

stg_bi['employment_type'] = stg_bi['employment_type_raw'].apply(normalize_bi_employment)

# ── Work model ────────────────────────────────────────────────────────────────
# jobLocationType = 'TELECOMMUTE' signals remote, absent = onsite
# No hybrid signal in Built In payload — if we see hybrid in title we catch it
def derive_bi_work_model(row) -> str:
    title = row['job_title'] or ''
    if row['job_location_type'] == 'TELECOMMUTE': return 'remote'
    if HYBRID_RE.search(title):                   return 'hybrid'
    if REMOTE_RE.search(title):                   return 'remote'
    return 'onsite'

stg_bi['work_model'] = stg_bi.apply(derive_bi_work_model, axis=1)

# ── Salary — annual only, with sanity filter for mislabeled hourly rates ──────
# Built In occasionally tags hourly rates as YEAR (e.g. $60/$75 at Insomniac)
# Null out anything below $1,000 annual as implausible
MIN_PLAUSIBLE_ANNUAL = 1000

stg_bi['salary_min'] = stg_bi.apply(
    lambda r: r['salary_min_raw']
    if r['salary_unit'] == 'YEAR'
    and isinstance(r['salary_min_raw'], (int, float))
    and r['salary_min_raw'] >= MIN_PLAUSIBLE_ANNUAL
    else None, axis=1
)
stg_bi['salary_max'] = stg_bi.apply(
    lambda r: r['salary_max_raw']
    if r['salary_unit'] == 'YEAR'
    and isinstance(r['salary_max_raw'], (int, float))
    and r['salary_max_raw'] >= MIN_PLAUSIBLE_ANNUAL
    else None, axis=1
)

print("employment_type value counts:")
print(stg_bi['employment_type'].value_counts(dropna=False))
print("\nwork_model value counts:")
print(stg_bi['work_model'].value_counts(dropna=False))
print("\nSalary coverage:")
print(f"  salary_min populated: {stg_bi['salary_min'].notna().sum()} / {len(stg_bi)}")
print("\nRows nulled out by sanity filter:")
print(stg_bi[
    (stg_bi['salary_unit'] == 'YEAR') &
    (stg_bi['salary_min_raw'].notna()) &
    (stg_bi['salary_min'].isna())
][['job_title', 'salary_min_raw', 'salary_max_raw']])

employment_type value counts:
employment_type
full_time    18
Name: count, dtype: int64

work_model value counts:
work_model
onsite    12
remote     6
Name: count, dtype: int64

Salary coverage:
  salary_min populated: 13 / 18

Rows nulled out by sanity filter:
               job_title  salary_min_raw  salary_max_raw
8  Research Data Analyst            60.0            75.0


In [44]:
# ── Senior title filter ───────────────────────────────────────────────────────
SENIOR_RE = re.compile(
    r'\b(senior|sr\.?|lead|principal|staff|manager|director|vp|vice president|'
    r'avp|head of|architect|chief|svp|evp|gvp|president|officer|executive|leader)\b',
    re.IGNORECASE
)

stg_bi['is_senior'] = stg_bi['job_title'].apply(
    lambda t: bool(SENIOR_RE.search(t)) if t else False
)

print(f"Senior titles flagged: {stg_bi['is_senior'].sum()} / {len(stg_bi)}")
print(stg_bi[stg_bi['is_senior']]['job_title'].tolist())

Senior titles flagged: 1 / 18
['Senior Data Analyst']


In [45]:
# ── Dedup within Built In ─────────────────────────────────────────────────────
before = len(stg_bi)
stg_bi = (
    stg_bi.sort_values('ingested_at', ascending=False)
          .drop_duplicates(subset='job_id', keep='first')
          .reset_index(drop=True)
)
print(f"Rows before dedup: {before} | after: {len(stg_bi)} | removed: {before - len(stg_bi)}")

Rows before dedup: 18 | after: 18 | removed: 0


In [46]:
# ── Apply filters and select final columns ────────────────────────────────────
stg_bi_final = (
    stg_bi[~stg_bi['is_senior']]
    [[
        'job_id', 'source', 'job_title', 'company_name', 'job_url',
        'date_posted', 'description', 'city', 'state', 'country',
        'latitude', 'longitude', 'work_model', 'employment_type',
        'salary_min', 'salary_max', 'ingested_at',
    ]]
    .reset_index(drop=True)
)

print(f"Final stg_builtin rows: {len(stg_bi_final)}")
print(f"\nNull rates:")
print((stg_bi_final.isna().sum() / len(stg_bi_final) * 100).round(1).to_string())
print(f"\nSample output:")
display(stg_bi_final.head(10))

Final stg_builtin rows: 17

Null rates:
job_id              0.0
source              0.0
job_title           0.0
company_name        0.0
job_url             0.0
date_posted         0.0
description         0.0
city               41.2
state              41.2
country             5.9
latitude           41.2
longitude          41.2
work_model          0.0
employment_type     0.0
salary_min         29.4
salary_max         29.4
ingested_at         0.0

Sample output:


,job_id,source,job_title,company_name,job_url,date_posted,description,city,state,country,latitude,longitude,work_model,employment_type,salary_min,salary_max,ingested_at
0,9598250,builtin,Data Analyst,Whalar,https://www.builtinnyc.com/job/data-analyst/9598250,2026-06-03,"Job Title: Data Analyst Location: New York, NY Start Date : ASAP Whalar is t...",New York,New York,USA,40.713047,-74.00723,onsite,full_time,55000.0,62500.0,2026-06-04 12:04:35.529167+00:00
1,9604259,builtin,Legal Assistant / Data Analyst Supporting the US Attorney's Office,"Compass Strategy Solutions, LLC",https://www.builtinnyc.com/job/legal-assistant-data-analyst-supporting-us-at...,2026-06-03,Job Summary & Responsibilities Be a part of the nationwide law enforcement i...,New York,New York,USA,40.713047,-74.00723,onsite,full_time,NaN,NaN,2026-06-04 12:04:35.529167+00:00
2,9591499,builtin,Data Scientist Analyst - Secondaries & Primaries,Ardian,https://www.builtinnyc.com/job/data-scientist-analyst-secondaries-primaries/...,2026-06-02,The Role This position offers a unique opportunity to immerse yourself in th...,New York,New York,USA,40.713047,-74.00723,onsite,full_time,NaN,NaN,2026-06-04 12:04:35.529167+00:00
3,9599913,builtin,Data Analyst,QuinStreet,https://www.builtinnyc.com/job/data-analyst/9599913,2026-06-03,Powering Performance Marketplaces in Digital Media QuinStreet is a pioneer i...,NaN,NaN,USA,NaN,NaN,remote,full_time,85000.0,100000.0,2026-06-04 12:04:35.529167+00:00
4,9593023,builtin,Research Data Analyst,Insomniac Games,https://www.builtinnyc.com/job/research-data-analyst/9593023,2026-06-02,Research Data Analyst - 6 Month CONTRACT Insomniac Games is looking for a ne...,NaN,NaN,USA,NaN,NaN,remote,full_time,NaN,NaN,2026-06-04 12:04:35.529167+00:00
5,9593070,builtin,Data Analyst - Economic Insights & Communications,Stripe,https://www.builtinnyc.com/job/data-analyst-economic-insights-communications...,2026-06-03,Data Analyst - Economic Insights & Communications Who we are About Stripe St...,NaN,NaN,USA,NaN,NaN,remote,full_time,NaN,NaN,2026-06-04 12:04:35.529167+00:00
6,9603111,builtin,Technical Business Analyst - User Success & Enablement | Enterprise Data Man...,Neuberger Berman,https://www.builtinnyc.com/job/technical-business-analyst-user-success-enabl...,2026-06-03,Neuberger is seeking a Business Analyst to join the User Success and Enablem...,New York,New York,USA,40.713047,-74.00723,onsite,full_time,95000.0,105000.0,2026-06-04 12:04:35.529167+00:00
7,9564259,builtin,Data Analyst,iHeartMedia,https://www.builtinnyc.com/job/data-analyst/9564259,2026-06-01,iHeartMedia Current employees and contingent workers click here to apply and...,New York,New York,USA,40.713047,-74.00723,onsite,full_time,80000.0,100000.0,2026-06-04 12:04:35.529167+00:00
8,9562534,builtin,Supplier Data Operations Analyst,Synapse Health,https://www.builtinnyc.com/job/supplier-data-operations-analyst/9562534,2026-06-01,"Who We Are : At Synapse Health, we're streamlining the durable medical equip...",NaN,NaN,USA,NaN,NaN,remote,full_time,68800.0,86000.0,2026-06-04 12:04:35.529167+00:00
9,9569462,builtin,Data Analyst,Queens Public Library,https://www.builtinnyc.com/job/data-analyst/9569462,2026-06-01,Queens Public Library is a national and international leader in the delivery...,Queens,New York,USA,40.681490,-73.83652,onsite,full_time,65000.0,75000.0,2026-06-04 12:04:35.529167+00:00


In [36]:
print(f"\nAll titles:")
for t in sorted(stg_bi_final['job_title'].tolist()):
    print(t)


All titles:
Data Analyst
Data Analyst
Data Analyst
Data Analyst
Data Analyst
Data Analyst
Data Analyst
Data Analyst - Economic Insights & Communications
Data Analyst and Reporting Specialist
Data Operations Analyst
Data Scientist Analyst - Secondaries & Primaries
Investment Quant & Data Analyst
Legal Assistant / Data Analyst Supporting the US Attorney's Office
Media Data Analyst
Research Data Analyst
Supplier Data Operations Analyst
Technical Business Analyst - User Success & Enablement | Enterprise Data Management


In [42]:
# Inspect salary_unit for all rows to confirm the YEAR filter is working
print(stg_bi[['job_title', 'salary_min_raw', 'salary_max_raw', 'salary_unit']].to_string())

                                                                              job_title  salary_min_raw  salary_max_raw salary_unit
0                                                                          Data Analyst         55000.0         62500.0        YEAR
1                                                                          Data Analyst         85000.0        100000.0        YEAR
2                                                                          Data Analyst         70000.0         90000.0        YEAR
3                                                                          Data Analyst         65000.0         75000.0        YEAR
4                                                      Supplier Data Operations Analyst         68800.0         86000.0        YEAR
5                                                                          Data Analyst         80000.0        100000.0        YEAR
6                    Legal Assistant / Data Analyst Supporting the US Attorn

## 4. Combined View

In [48]:
# ── Combined view ──────────────────────────────────────────────────
# Union all three staging outputs into one DataFrame
combined = pd.concat([stg_final, stg_ts_final, stg_bi_final], ignore_index=True)

print(f"Total rows by source:")
print(combined['source'].value_counts())
print(f"\nTotal combined rows: {len(combined)}")

Total rows by source:
source
jsearch       82
theirstack    18
builtin       17
Name: count, dtype: int64

Total combined rows: 117


In [49]:
# ── Cross-source deduplication ────────────────────────────────────────────────
# Same job appearing in multiple sources — match on normalized title + company
combined['dedup_key'] = (
    combined['job_title'].str.strip().str.lower()
    + ' | '
    + combined['company_name'].str.strip().str.lower()
)

# Find duplicates
cross_dupes = combined[combined.duplicated('dedup_key', keep=False)].sort_values('dedup_key')
print(f"Cross-source duplicate groups found: {cross_dupes['dedup_key'].nunique()}")
print(f"Total rows involved: {len(cross_dupes)}")
print()
display(cross_dupes[['job_title', 'company_name', 'source', 'dedup_key']].reset_index(drop=True))

Cross-source duplicate groups found: 5
Total rows involved: 10



,job_title,company_name,source,dedup_key
0,"Analytics Engineer, Service Ops Analytics & AI",Capgemini,jsearch,"analytics engineer, service ops analytics & ai | capgemini"
1,"Analytics Engineer, Service Ops Analytics & AI",Capgemini,theirstack,"analytics engineer, service ops analytics & ai | capgemini"
2,Data Analyst - Economic Insights & Communications,Stripe,jsearch,data analyst - economic insights & communications | stripe
3,Data Analyst - Economic Insights & Communications,Stripe,builtin,data analyst - economic insights & communications | stripe
4,"Data Analyst, Operations Planning",Lyft,jsearch,"data analyst, operations planning | lyft"
5,"Data Analyst, Operations Planning",Lyft,jsearch,"data analyst, operations planning | lyft"
6,Data Scientist Analyst - Secondaries & Primaries,Ardian,jsearch,data scientist analyst - secondaries & primaries | ardian
7,Data Scientist Analyst - Secondaries & Primaries,Ardian,builtin,data scientist analyst - secondaries & primaries | ardian
8,Junior Data Analyst w/ Workday Exp....Rate-$30/hronC2C....Locals Only,Jobs via Dice,jsearch,junior data analyst w/ workday exp....rate-$30/hronc2c....locals only | jobs...
9,Junior Data Analyst w/ Workday Exp....Rate-$30/hronC2C....Locals Only,Jobs via Dice,theirstack,junior data analyst w/ workday exp....rate-$30/hronc2c....locals only | jobs...


In [50]:
# Keep one row per dedup_key — prefer builtin > theirstack > jsearch
# (Built In tends to have the cleanest structured salary data)
source_priority = {'builtin': 0, 'theirstack': 1, 'jsearch': 2}
combined['source_rank'] = combined['source'].map(source_priority)

before = len(combined)
combined_deduped = (
    combined.sort_values(['dedup_key', 'source_rank'])
            .drop_duplicates(subset='dedup_key', keep='first')
            .drop(columns=['dedup_key', 'source_rank'])
            .reset_index(drop=True)
)
after = len(combined_deduped)

print(f"Rows before cross-source dedup: {before}")
print(f"Rows after cross-source dedup:  {after}")
print(f"Duplicates removed: {before - after}")
print(f"\nFinal row counts by source:")
print(combined_deduped['source'].value_counts())

Rows before cross-source dedup: 117
Rows after cross-source dedup:  112
Duplicates removed: 5

Final row counts by source:
source
jsearch       77
theirstack    18
builtin       17
Name: count, dtype: int64


In [51]:
# ── Combined null rates ───────────────────────────────────────────────────────
print("Null rates across combined dataset:")
null_rates = (combined_deduped.isna().sum() / len(combined_deduped) * 100).round(1)
print(null_rates.to_string())

Null rates across combined dataset:
job_id              0.0
source              0.0
job_title           0.0
company_name        0.0
job_url             0.0
date_posted         0.0
description         0.0
city               11.6
state              11.6
country             6.2
latitude           11.6
longitude          11.6
work_model          0.0
employment_type     1.8
salary_min         72.3
salary_max         72.3
ingested_at         0.0


In [52]:
# ── Distribution checks ───────────────────────────────────────────────────────
print("work_model distribution:")
print(combined_deduped['work_model'].value_counts())

print("\nemployment_type distribution:")
print(combined_deduped['employment_type'].value_counts(dropna=False))

print("\nsalary coverage:")
total = len(combined_deduped)
has_salary = combined_deduped['salary_min'].notna().sum()
print(f"  {has_salary} / {total} rows have salary ({round(has_salary/total*100, 1)}%)")

print("\nsalary range where populated:")
sal = combined_deduped[combined_deduped['salary_min'].notna()]
print(f"  min: ${sal['salary_min'].min():,.0f}")
print(f"  max: ${sal['salary_max'].max():,.0f}")
print(f"  median min: ${sal['salary_min'].median():,.0f}")

work_model distribution:
work_model
onsite    80
remote    22
hybrid    10
Name: count, dtype: int64

employment_type distribution:
employment_type
full_time    97
contract      9
other         3
NaN           2
part_time     1
Name: count, dtype: int64

salary coverage:
  31 / 112 rows have salary (27.7%)

salary range where populated:
  min: $50,000
  max: $225,000
  median min: $85,000


In [56]:
# ── Join coverage — how many combined rows have enrichment data? ───────────────
enrichment_df = run_query("SELECT * FROM ENRICHED.PUBLIC.JOB_ENRICHMENT")
print(f"Enrichment records loaded: {len(enrichment_df)}")

enrich_ids = set(enrichment_df['JOB_ID'].astype(str))
combined_deduped['has_enrichment'] = combined_deduped['job_id'].astype(str).isin(enrich_ids)

total = len(combined_deduped)
matched = combined_deduped['has_enrichment'].sum()
print(f"\nEnrichment join coverage: {matched} / {total} ({round(matched/total*100, 1)}%)")
print("(Note: Smaller total reflects post-filter, post-dedup count — not raw ingestion count)")
print(f"\nBy source:")
print(combined_deduped.groupby('source')['has_enrichment'].value_counts().unstack(fill_value=0))

Enrichment records loaded: 172

Enrichment join coverage: 112 / 112 (100.0%)
(Note: Smaller total reflects post-filter, post-dedup count — not raw ingestion count)

By source:
has_enrichment  True
source              
builtin           17
jsearch           77
theirstack        18


In [57]:
# ── Salary coverage: structured payload vs LLM enrichment ────────────────────
# Join enrichment salary fields onto combined_deduped
enrich_salary = enrichment_df[['JOB_ID', 'SALARY_MIN', 'SALARY_MAX']].copy()
enrich_salary['JOB_ID'] = enrich_salary['JOB_ID'].astype(str)

combined_with_enrich = combined_deduped.merge(
    enrich_salary,
    left_on='job_id',
    right_on='JOB_ID',
    how='left'
).rename(columns={
    'salary_min':  'payload_salary_min',
    'salary_max':  'payload_salary_max',
    'SALARY_MIN':  'llm_salary_min',
    'SALARY_MAX':  'llm_salary_max',
})

total = len(combined_with_enrich)
payload_has  = combined_with_enrich['payload_salary_min'].notna().sum()
llm_has      = combined_with_enrich['llm_salary_min'].notna().sum()

# Combined: use payload salary first, fall back to LLM
combined_with_enrich['final_salary_min'] = combined_with_enrich['payload_salary_min'].fillna(
    combined_with_enrich['llm_salary_min']
)
combined_with_enrich['final_salary_max'] = combined_with_enrich['payload_salary_max'].fillna(
    combined_with_enrich['llm_salary_max']
)

final_has = combined_with_enrich['final_salary_min'].notna().sum()

print(f"Salary coverage across {total} rows:")
print(f"  Structured payload only:      {payload_has} / {total} ({round(payload_has/total*100, 1)}%)")
print(f"  LLM enrichment only:          {llm_has} / {total} ({round(llm_has/total*100, 1)}%)")
print(f"  Combined (payload + LLM gap): {final_has} / {total} ({round(final_has/total*100, 1)}%)")
print(f"  LLM filled in gaps:           +{final_has - payload_has} rows")

Salary coverage across 112 rows:
  Structured payload only:      31 / 112 (27.7%)
  LLM enrichment only:          62 / 112 (55.4%)
  Combined (payload + LLM gap): 71 / 112 (63.4%)
  LLM filled in gaps:           +40 rows


In [58]:
# ── Where did the LLM add salary that payload didn't have? ───────────────────
llm_filled = combined_with_enrich[
    combined_with_enrich['payload_salary_min'].isna() &
    combined_with_enrich['llm_salary_min'].notna()
][['job_title', 'company_name', 'source', 'llm_salary_min', 'llm_salary_max']]

print(f"Jobs where LLM extracted salary but payload had none ({len(llm_filled)} rows):")
display(llm_filled.reset_index(drop=True))

Jobs where LLM extracted salary but payload had none (40 rows):


,job_title,company_name,source,llm_salary_min,llm_salary_max
0,"[P] Data Scientist, Policy",Anthropic,jsearch,275000.0,370000.0
1,Advertising Data Analyst,Hearst Television,theirstack,75000.0,95000.0
2,"Analyst, Strategy & Operations",Mastercard,jsearch,85000.0,137000.0
3,"Analytics Engineer (TakeUp), Mid-Level (Remote)",jobright.com,jsearch,170000.0,720000.0
4,Azure Databricks Platform Engineer,Fiserv,jsearch,128000.0,216000.0
5,Business Analyst III AMZ26810.1,Amazon Web Services (AWS),jsearch,131942.0,185000.0
6,"Business Intelligence Engineer, Ad Sales Finance Analytics",Amazon,jsearch,99500.0,185000.0
7,"Business Intelligence Engineer, Rapid & Rural Logistics (R2L) Science & AI",Amazon,jsearch,99500.0,185000.0
8,Capital Markets Data Engineer: Real-Time Analytics & Automation,AmeriSave Mortgage,jsearch,95000.0,135000.0
9,"Credit Business Analyst, Growth",Robinhood,jsearch,119000.0,180000.0


In [59]:
# ── Final salary range after combining both sources ───────────────────────────
sal = combined_with_enrich[combined_with_enrich['final_salary_min'].notna()]
print(f"Salary range across {len(sal)} rows with salary data:")
print(f"  min:         ${sal['final_salary_min'].min():,.0f}")
print(f"  max:         ${sal['final_salary_max'].max():,.0f}")
print(f"  median min:  ${sal['final_salary_min'].median():,.0f}")
print(f"  median max:  ${sal['final_salary_max'].median():,.0f}")

print(f"\nBy source:")
display(
    sal.groupby('source').agg(
        count=('final_salary_min', 'count'),
        median_min=('final_salary_min', 'median'),
        median_max=('final_salary_max', 'median'),
    ).round(0)
)

Salary range across 71 rows with salary data:
  min:         $2,080
  max:         $720,000
  median min:  $85,000
  median max:  $120,000

By source:


,count,median_min,median_max
source,,,
builtin,13,80000.0,100000.0
jsearch,49,95000.0,138000.0
theirstack,9,75000.0,90000.0


In [60]:
# # ── Salary sanity check on LLM-extracted values ───────────────────────────────
# MIN_PLAUSIBLE = 1000
# MAX_PLAUSIBLE = 500000

# flagged = combined_with_enrich[
#     combined_with_enrich['llm_salary_min'].notna() & (
#         (combined_with_enrich['llm_salary_min'] < MIN_PLAUSIBLE) |
#         (combined_with_enrich['llm_salary_max'] > MAX_PLAUSIBLE)
#     )
# ][['job_title', 'company_name', 'source', 'llm_salary_min', 'llm_salary_max']]

# print(f"LLM salary values outside plausible range ({MIN_PLAUSIBLE:,} – {MAX_PLAUSIBLE:,}):")
# display(flagged.reset_index(drop=True))

LLM salary values outside plausible range (1,000 – 500,000):


,job_title,company_name,source,llm_salary_min,llm_salary_max
0,"Analytics Engineer (TakeUp), Mid-Level (Remote)",jobright.com,jsearch,170000.0,720000.0


In [62]:
# # ── Salary distribution — help decide where to draw the ceiling ───────────────
# sal_rows = combined_with_enrich[combined_with_enrich['final_salary_min'].notna()].copy()
# sal_rows = sal_rows.sort_values('final_salary_max', ascending=False)

# print("Top 20 salary rows by max salary:")
# display(sal_rows[['job_title', 'company_name', 'source', 
#                    'final_salary_min', 'final_salary_max']].head(20).reset_index(drop=True))

# print(f"\nRows that would be cut by a $500k ceiling:")
# cut = sal_rows[sal_rows['final_salary_max'] > 500000]
# display(cut[['job_title', 'company_name', 'final_salary_min', 'final_salary_max']].reset_index(drop=True))

# print(f"\nRows that would be cut by a $1k floor:")
# cut_floor = sal_rows[sal_rows['final_salary_min'] < 1000]
# display(cut_floor[['job_title', 'company_name', 'final_salary_min', 'final_salary_max']].reset_index(drop=True))

Top 20 salary rows by max salary:


,job_title,company_name,source,final_salary_min,final_salary_max
0,"Analytics Engineer (TakeUp), Mid-Level (Remote)",jobright.com,jsearch,170000.0,720000.0
1,Security Analytics Engineer L4 35,Netflix,jsearch,250000.0,421000.0
2,"[P] Data Scientist, Policy",Anthropic,jsearch,275000.0,370000.0
3,Data Analyst | Remote,Crossing Hurdles,jsearch,145600.0,249600.0
4,Data Quality Analyst – Enterprise Data,Bloomberg,jsearch,110000.0,225000.0
5,Azure Databricks Platform Engineer,Fiserv,jsearch,128000.0,216000.0
6,Data Engineer – IBM Quantum,IBM,jsearch,116000.0,200000.0
7,"Data Engineer, AWS GDSP A&I",Amazon Web Services (AWS),jsearch,132100.0,196600.0
8,Business Analyst III AMZ26810.1,Amazon Web Services (AWS),jsearch,131942.0,185000.0
9,"Business Intelligence Engineer, Ad Sales Finance Analytics",Amazon,jsearch,99500.0,185000.0



Rows that would be cut by a $500k ceiling:


,job_title,company_name,final_salary_min,final_salary_max
0,"Analytics Engineer (TakeUp), Mid-Level (Remote)",jobright.com,170000.0,720000.0



Rows that would be cut by a $1k floor:


,job_title,company_name,final_salary_min,final_salary_max


### Full Join & Preview

In [69]:
# ── Full join — combined staging + all enrichment fields ─────────────────────
enrich_clean = enrichment_df.copy()
enrich_clean.columns = [c.lower() for c in enrich_clean.columns]
enrich_clean['job_id'] = enrich_clean['job_id'].astype(str)

# Pull LLM salary out separately before join to avoid column conflicts
llm_salary = enrich_clean[['job_id', 'salary_min', 'salary_max']].rename(columns={
    'salary_min': 'llm_salary_min',
    'salary_max': 'llm_salary_max',
})

# Drop salary and source from enrichment — salary handled separately, source redundant
enrich_no_salary = enrich_clean.drop(columns=['salary_min', 'salary_max', 'source'])

# Drop temporary has_enrichment column from earlier check
full_join = combined_deduped.drop(columns=['has_enrichment'], errors='ignore').merge(
    enrich_no_salary,
    on='job_id',
    how='left'
).merge(
    llm_salary,
    on='job_id',
    how='left'
)

# Payload salary first, LLM fills gaps
full_join['final_salary_min'] = full_join['salary_min'].fillna(full_join['llm_salary_min'])
full_join['final_salary_max'] = full_join['salary_max'].fillna(full_join['llm_salary_max'])

print(f"Full join rows: {len(full_join)}")
print(f"Columns ({len(full_join.columns)}): {full_join.columns.tolist()}")
print(f"\nSalary coverage after combining payload + LLM:")
print(f"  {full_join['final_salary_min'].notna().sum()} / {len(full_join)} ({round(full_join['final_salary_min'].notna().sum()/len(full_join)*100,1)}%)")

Full join rows: 112
Columns (36): ['job_id', 'source', 'job_title', 'company_name', 'job_url', 'date_posted', 'description', 'city', 'state', 'country', 'latitude', 'longitude', 'work_model', 'employment_type', 'salary_min', 'salary_max', 'ingested_at', 'inferred_seniority', 'is_title_inflated', 'inflation_reasoning', 'role_archetype', 'work_focus', 'tech_stack_required', 'tech_stack_preferred', 'paradigms_required', 'paradigms_preferred', 'degree_requirement', 'years_required_min', 'years_required_max', 'confidence_score', 'enriched_at', 'model_version', 'llm_salary_min', 'llm_salary_max', 'final_salary_min', 'final_salary_max']

Salary coverage after combining payload + LLM:
  71 / 112 (63.4%)


In [70]:
# ── Select final mart columns ─────────────────────────────────────────────────
fct = full_join[[
    # --- identity ---
    'job_id',
    'source',

    # --- job info ---
    'job_title',
    'company_name',
    'job_url',
    'date_posted',

    # --- location ---
    'city',
    'state',
    'country',
    'latitude',
    'longitude',
    'work_model',

    # --- employment ---
    'employment_type',

    # --- salary ---
    'final_salary_min',
    'final_salary_max',

    # --- enrichment: classification ---
    'inferred_seniority',
    'role_archetype',
    'work_focus',
    'is_title_inflated',
    'inflation_reasoning',

    # --- enrichment: requirements ---
    'tech_stack_required',
    'tech_stack_preferred',
    'paradigms_required',
    'paradigms_preferred',
    'degree_requirement',
    'years_required_min',
    'years_required_max',

    # --- enrichment: metadata ---
    'confidence_score',
    'enriched_at',
    'ingested_at',
]].copy()

print(f"Final mart shape: {fct.shape[0]} rows x {fct.shape[1]} columns")

Final mart shape: 112 rows x 30 columns


In [71]:
# ── Final null rates ──────────────────────────────────────────────────────────
print("Null rates across final mart:")
print((fct.isna().sum() / len(fct) * 100).round(1).to_string())

Null rates across final mart:
job_id                   0.0
source                   0.0
job_title                0.0
company_name             0.0
job_url                  0.0
date_posted              0.0
city                    11.6
state                   11.6
country                  6.2
latitude                11.6
longitude               11.6
work_model               0.0
employment_type          1.8
final_salary_min        36.6
final_salary_max        38.4
inferred_seniority       0.0
role_archetype           0.0
work_focus               0.0
is_title_inflated        0.0
inflation_reasoning     84.8
tech_stack_required      0.0
tech_stack_preferred     0.0
paradigms_required       0.0
paradigms_preferred      0.0
degree_requirement       0.0
years_required_min      23.2
years_required_max      50.0
confidence_score         0.0
enriched_at              0.0
ingested_at              0.0


In [72]:
# ── Final sample ──────────────────────────────────────────────────────────────
display(fct.head(20))

,job_id,source,job_title,company_name,job_url,date_posted,city,state,country,latitude,...,tech_stack_required,tech_stack_preferred,paradigms_required,paradigms_preferred,degree_requirement,years_required_min,years_required_max,confidence_score,enriched_at,ingested_at
0,6vi0NY-dMVCh0ld7AAAAAA==,jsearch,"[P] Data Scientist, Policy",Anthropic,https://www.linkedin.com/jobs/view/p-data-scientist-policy-at-anthropic-4424...,2026-06-03,New York,New York,US,40.712775,...,"[\n ""python"",\n ""sql""\n]",[],"[\n ""causal inference"",\n ""data analysis""\n]",[],equivalent_ok,6.0,NaN,0.85,2026-06-04 12:11:02.146010+00:00,2026-06-04 12:04:35.529025+00:00
1,ap_u4Cqrk1EK1nA3AAAAAA==,jsearch,Adobe Customer Journey Analytics Engineer - Remote 35,Nava Software Solutions,https://www.jobilize.com/job/us-ny-all-cities-adobe-customer-journey-analyti...,2026-06-01,New York,New York,US,40.712775,...,"[\n ""adobe experience platform"",\n ""adobe customer journey analytics""\n]",[],"[\n ""data schema design"",\n ""data management"",\n ""customer journey analyt...",[],none,NaN,NaN,0.85,2026-06-04 12:09:04.304544+00:00,2026-06-04 12:04:35.529025+00:00
2,698550590,theirstack,Advertising Data Analyst,Hearst Television,https://www.linkedin.com/jobs/view/advertising-data-analyst-at-hearst-televi...,2026-05-31,New York,NY,US,40.714270,...,"[\n ""power bi"",\n ""excel""\n]",[],"[\n ""data integration"",\n ""performance analysis"",\n ""data visualization""\n]",[],bachelors,2.0,NaN,0.90,2026-06-03 22:32:39.264563+00:00,2026-05-31 18:11:41.996056+00:00
3,MWNp5rzUlOQyQa2WAAAAAA==,jsearch,"Analyst, Strategy & Operations",Mastercard,https://careers.mastercard.com/us/en/job/R-278974/Analyst-Strategy-Operations,2026-06-04,New York,New York,US,40.712775,...,"[\n ""excel"",\n ""power bi""\n]","[\n ""tableau"",\n ""salesforce""\n]","[\n ""data analysis"",\n ""performance tracking"",\n ""reporting""\n]","[\n ""data-driven insights"",\n ""sales strategy""\n]",bachelors,NaN,NaN,0.90,2026-06-04 12:11:52.022807+00:00,2026-06-04 12:04:35.529025+00:00
4,sgF_XtGH7nCvIqGiAAAAAA==,jsearch,"Analytics Engineer (TakeUp), Mid-Level (Remote)",jobright.com,https://www.monster.com/job-openings/analytics-engineer-takeup-mid-level-rem...,2026-06-02,NaN,NaN,NaN,NaN,...,"[\n ""sql"",\n ""streamlit"",\n ""flask"",\n ""fastapi"",\n ""dash"",\n ""panel"",...","[\n ""snowflake""\n]","[\n ""data modeling"",\n ""ci/cd"",\n ""data quality"",\n ""analytics pipelines...","[\n ""ml workflows""\n]",none,NaN,NaN,0.85,2026-06-04 12:10:09.866280+00:00,2026-06-04 12:04:35.529025+00:00
5,Tbo8MNYTsx8gOrxhAAAAAA==,jsearch,Analytics Engineer - Sports Data & Modeling (Hybrid),Omaze,https://www.jobleads.com/us/job/analytics-engineer-sports-data-modeling-hybr...,2026-06-01,Jersey City,New Jersey,US,40.719534,...,"[\n ""sql""\n]",[],"[\n ""data modeling""\n]",[],none,2.0,5.0,0.85,2026-06-04 12:08:53.972668+00:00,2026-06-04 12:04:35.529025+00:00
6,703117190,theirstack,"Analytics Engineer, Service Ops Analytics & AI",Capgemini,https://www.linkedin.com/jobs/view/analytics-engineer-service-ops-analytics-...,2026-06-02,New York,NY,US,40.714270,...,"[\n ""sql"",\n ""python"",\n ""dbt"",\n ""snowflake"",\n ""git"",\n ""airflow""\n]",[],"[\n ""etl design"",\n ""data governance"",\n ""data quality"",\n ""data modelin...",[],none,NaN,NaN,0.90,2026-06-04 12:14:47.739068+00:00,2026-06-04 12:04:35.529034+00:00
7,pku1umkxPXBahuoPAAAAAA==,jsearch,Azure Databricks Platform Engineer,Fiserv,https://www.linkedin.com/jobs/view/azure-databricks-platform-engineer-at-fis...,2026-06-03,Berkeley Heights,New Jersey,US,40.670222,...,"[\n ""databricks"",\n ""azure data lake storage gen 2"",\n ""azure data explor...","[\n ""sql server""\n]","[\n ""data cleaning"",\n ""data transformation"",\n ""data lakehouse loading"",...","[\n ""cloud data migration""\n]",none,5.0,5.0,0.90,2026-06-04 12:09:56.035229+00:00,2026-06-04 12:04:35.529025+00:00
8,cCoTDP68ABEHKXGiAAAAAA==,jsearch,Billing Analyst: Data-Driven Invoicing & Compliance,"Intercontinental 

In [ ]:
# # ── Close connection ──────────────────────────────────────────────────────────
# conn.close()
# print('Connection closed.')